# 05 Mine pairs, train A3 (DPO)

Runs the A2 seed-0 adapter on dev rows (plus extra unused curated train rows), keeps its failures as rejected
answers against the curated gold as chosen, and writes `data/arms/A3_pairs.jsonl`. Then trains A3 with DPO
(TRL) starting from A2, with A2 as the reference model. Output: `checkpoints/A3-s0-{git_sha}/final/` with a
`dhanground.json` record. Manifest: that `dhanground.json` (records the A2 init it started from).

**Needs:** a GPU (L4), `requirements-gpu.txt`, `hf auth login`; `JUDGE_MODEL` (optional) adds LLM grading of failures.

In [ ]:
import json, os, sys
from collections import Counter
from pathlib import Path

# Run from the repo root so relative paths match the README runbook.
ROOT = Path.cwd()
while not (ROOT / "core").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
PY = sys.executable  # `!{PY} -m ...` runs the CLI with this kernel's interpreter
print("repo root:", ROOT)

In [ ]:
# Newest finished A2 seed-0 run (it has dhanground.json); set A2 to an HF repo id to use a pushed adapter.
done = sorted(Path("checkpoints").glob("A2-s0-*/dhanground.json"), key=lambda p: p.stat().st_mtime)
assert done, "no finished A2 seed-0 adapter under checkpoints/; run notebook 04 first"
A2 = str(done[-1].parent)
HUB_ORG = None
HUB_ARG = f"--hub-org {HUB_ORG}" if HUB_ORG else ""
print("A2 adapter:", A2)

In [ ]:
!{PY} -m intelligence.mine_pairs --a2 {A2} --extra-curated-train 1500

In [ ]:
!{PY} -m adapters.aligntune_dpo --a2 {A2} {HUB_ARG}

## Summary

In [ ]:
def read_jsonl(path):
    path = Path(path)
    return [json.loads(l) for l in path.open(encoding="utf-8")] if path.exists() else []


pairs = read_jsonl("data/arms/A3_pairs.jsonl")
print(f"{len(pairs)} pairs", dict(Counter(p["reject_reason"] for p in pairs)))
for p in pairs[:2]:
    print(f"- {p['row_id']} ({p['reject_reason']}): rejected = {p['rejected'][0]['content'][:120]}")
for rec in sorted(Path("checkpoints").glob("A3-s*/final/dhanground.json")):
    print(rec.parent.parent.name, json.loads(rec.read_text()))